# Reaction diagrams from rxflow results

Set `DB_PATH` to a reaction database. The notebook also reads `precursors.sqlite` and `encounters.sqlite` beside it and the shared `solution_references/references.sqlite` (both paths can be overridden below). Rerun all cells to pick up newly completed calculations. All databases are opened read-only. `ENERGY_MODE = "auto"` also checks `singlepoints.sqlite` in the run and shared-reference directories.

The extended diagram follows **protonated ligand–metal complex + fully protonated free substrate + acetate → deprotonated ligand–metal complex + free substrate + acetic acid → relaxed encounter complex + acetic acid → bound reactant + acetic acid → TS + acetic acid → proton-transfer product + acetic acid**. “LM” means the ligand–metal complex; AcOH means acetic acid. Each state has total charge +1 and the same atoms. The substrate is already inside the encounter complex, bound reactant, TS and product, so it is added separately only to the first two states. Encounter geometries start from ligand–metal + substrate embeddings and relax freely: fragments may separate or change bonding, with no geometry filter or separation limit. Converged minima with zero imaginary frequencies supply this level. The ligand-deprotonation step is a thermodynamic difference; no acetate-mediated deprotonation TS was calculated.

For each ligand, the zero is the **lowest completed protonated LM + substrate + acetate total across its recorded binding permutations and native protonation sites**. Every site and pathway for that ligand uses this same absolute reference. If no protonated starting total is available, the notebook explicitly falls back to the lowest bound reactant + acetic acid. If acetic acid is not available at all, it shows encounter–reactant–TS–product levels relative to the lowest reactant; it never mixes unbalanced and acid-balanced totals. Missing precursor levels remain blank and no line is drawn across a missing level.

Each species/state uses its lowest **successful NumFreq Gibbs energy**, including conformers outside the electronic-energy retained set. Pending and failed candidates do not erase completed results; results may improve while jobs finish. Tables list the chosen job/conformer, absolute energy, job statuses and any missing inputs outside the plot. Energies on the diagram are in kcal/mol. They use the existing ORCA thermochemistry convention without an additional solution standard-state or concentration correction.

Coverage tables include every recorded substrate site. Expected permutations are the union recorded for that ligand in the reaction database; arrangements removed before preparation and ligands absent from the database are outside this count. Coverage counts complete reactant–TS–product pathways independently of precursor readiness. Every complete reaction pair/resonance remains separate, even if several routes share a precursor energy. Native ligand protonation sites are also kept separate if more than one is recorded; the TS H-acceptor does not select a precursor protonation site.

Before the profiles, two substrate drawings highlight **Site A** and **Site B**, respectively. These apply to all ligands in the run.

Energy plots have tilted state labels and no text or numerical annotations inside the axes. Titles state the active energy model. Compact legends sit outside and give site, permutation, pair, and forward/reverse barriers; native protonation-site details remain in the tables. Below each plot, RDKit draws each reaction pathway's ligand: **blue** highlights metal-binding atoms, **red** the actual TS H-acceptor, and **purple** their overlap. The notebook also renders the lowest-reactant pathway's optimized structures.

The proton-transfer barriers are **forward = G(TS) − min[G(encounter), G(bound reactant)]** and **reverse = G(TS) − G(bound product)**. A pathway qualifies only if **both are ≤ 25 kcal/mol**, using the unrounded energies. Distinct, contrasting plot colors identify each substrate site and barrier class; qualifying pathways are listed in a separate report. Every plot legend and ligand-grid caption includes both barrier values. Ligand-grid captions also give the bound reactant’s relative Gibbs energy: G(bound deprotonated reactant for this group) − the lowest successful bound-reactant G across comparable groups for that ligand, including groups without a TS. This reference uses the active energy model and keeps substrate, charge and multiplicity separate. Tables have no colored shading, and ligand-grid captions use black text. The grid section headings identify pathways passing both limits; the plot legends give the barriers without a threshold tag. The blue/red/purple atom highlights retain their chemical meaning. The forward reference uses the encounter minimum for the same ligand–metal binding arrangement when it is lower; otherwise it uses the bound reactant. If no matching encounter energy is available in the active model, it uses the bound reactant. The ranked report, threshold, plot legends and grid captions all use this definition; the reverse reference remains the product.

**DFT single-point correction:** `rxflow singlepoint` calculates PBE0/def2-TZVP-D4 with SMD(water) on the selected gxTB geometries. For each component, the plotted correction is **G(corrected) = G(gxTB NumFreq) − E(gxTB NumFreq) + E(PBE0-D4/SMD)**. The electronic and Gibbs gxTB energies must belong to the same saved geometry and NumFreq result. Single points do not optimize or recalculate frequencies. Only the lowest-G(gxTB) geometry per state is selected; other conformers are not ranked at DFT.

In **auto** mode the whole run stays on gxTB energies until every currently selected reaction state, precursor, substrate, acetic acid and acetate has a matching successful single point. It then switches all plotted components together, including the common reference and forward/reverse barriers. A changed source geometry or gxTB result invalidates an old correction. `ENERGY_MODE = "gxtb"` forces the original energies; `"dft"` explicitly shows partial corrected results with missing states left unavailable. The notebook reports the active model and completion count above the plots.

The acid/base workflow also freshly embeds and optimizes the **protonated bound reactant (+2)**. Its balanced total is **G(protonated bound reactant) + G(acetate)**, with no extra substrate or acetic-acid term. When these calculations are prepared, the diagram shows two alternative formation routes: deprotonation before substrate binding, or substrate binding before deprotonation. Both join the existing bound reactant + AcOH state; the alternative intermediates are not connected to each other. Missing new levels remain blank. No acetate-mediated transition state is implied.


In [ ]:
from pathlib import Path
from html import escape
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor
import xyzrender
from IPython.display import display, Markdown, HTML
from hpc_precursors import load_diagram_inputs

DB_PATH = Path("hpc_tm_results/workflow.sqlite")  # change to your database
PRECURSOR_DB_PATH = DB_PATH.with_name("precursors.sqlite")
REFERENCE_DB_PATH = Path("solution_references/references.sqlite")  # shared acid/base results
ENERGY_MODE = "auto"  # auto: switch only when all selected single points are ready; or gxtb / dft
HARTREE_TO_KCAL = 627.509474
BARRIER_LIMIT_KCAL = 25.0  # Both forward and reverse proton-transfer barriers must pass.
TS_FAILURE_WINDOW_KCAL = 3.0  # Prioritize failed TS groups near each ligand's reactant minimum.

inputs = load_diagram_inputs(DB_PATH, PRECURSOR_DB_PATH, REFERENCE_DB_PATH, energy_mode=ENERGY_MODE)
jobs = inputs["jobs"]
precursor_jobs = inputs["precursor_jobs"]
reference_jobs = inputs["reference_jobs"]
bound_reactant_jobs = inputs["bound_reactant_jobs"]
encounter_jobs = inputs["encounter_jobs"]
precursor_references = inputs["diagram_references"]
reference_minima = inputs["reference_minima"]
acid_gibbs_eh = reference_minima.get("acetic_acid", {}).get("gibbs_free_energy_eh")
print(f"Loaded {len(encounter_jobs)} encounter-complex rows from {DB_PATH.with_name("encounters.sqlite")}")
print(f"Loaded {len(jobs)} reaction rows from {DB_PATH}")
print(f"Loaded {len(precursor_jobs)} precursor rows from {PRECURSOR_DB_PATH}")
print(f"Loaded {len(reference_jobs)} shared reference rows from {REFERENCE_DB_PATH}")
print(f"Loaded {len(bound_reactant_jobs)} protonated bound-reactant rows from {DB_PATH.with_name("acid_base.sqlite")}")
singlepoint_progress = inputs["singlepoint_summary"]
display(Markdown(f"**Plotted energy model:** {inputs['energy_model']}. "
                 f"**DFT single points:** {singlepoint_progress['ready']} / {singlepoint_progress['selected_states']} selected states ready."))
if ENERGY_MODE == "auto" and singlepoint_progress["active_mode"] == "gxtb":
    display(Markdown("Using gxTB energies for this run until all selected single points, including shared acid/base references, are complete. No partial DFT/gxTB mixture is plotted."))
if ENERGY_MODE == "dft" and singlepoint_progress["ready"] < singlepoint_progress["selected_states"]:
    display(Markdown("**Partial DFT view:** states without their matching single point are unavailable. No gxTB energies are substituted for them."))
counts = Counter((row["endpoint"], row["status"]) for row in jobs)
display([{ "endpoint": endpoint, "status": status, "rows": count }
         for (endpoint, status), count in sorted(counts.items())])
if not PRECURSOR_DB_PATH.is_file():
    display(Markdown("**Precursor database is not available.** Completed reaction pathways remain visible."))
if acid_gibbs_eh is None:
    display(Markdown("**Acetic-acid Gibbs energy is not available.** Showing three-state reaction diagrams in the active energy model until it finishes."))
elif "acetate" not in reference_minima:
    display(Markdown("**Acetate Gibbs energy is not available.** Protonated starting levels remain blank; each ligand uses its lowest bound reactant + acetic acid as the temporary reference."))


In [ ]:
# One substrate-site key for the run, before all pathway profiles.
def substrate_site_key(smiles, recorded_sites):
    """Use the same atom order and aromatic C-H symmetry classes as rxflow."""
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError(f"Invalid substrate SMILES: {smiles}")
    for atom in molecule.GetAtoms():
        atom.SetAtomMapNum(0)
    molecule = Chem.AddHs(molecule)
    ranks = Chem.CanonicalRankAtoms(molecule, breakTies=False)
    representatives = {}
    for atom in molecule.GetAtoms():
        if atom.GetIsAromatic() and atom.GetAtomicNum() == 6 and any(
                neighbor.GetAtomicNum() == 1 for neighbor in atom.GetNeighbors()):
            representatives.setdefault(ranks[atom.GetIdx()], atom.GetIdx())
    sites = []
    for index, (rank, representative) in enumerate(sorted(representatives.items(), key=lambda item: item[1])):
        # Spreadsheet-style names also cover substrates with more than 26 sites.
        number, name = index + 1, ""
        while number:
            number, remainder = divmod(number - 1, 26)
            name = chr(65 + remainder) + name
        members = [atom.GetIdx() for atom in molecule.GetAtoms()
                   if atom.GetAtomicNum() == 6 and ranks[atom.GetIdx()] == rank]
        sites.append(dict(name=name, substrate_site=representative + 1, atoms=members))
    known_sites = {site["substrate_site"] for site in sites}
    unknown_sites = set(recorded_sites) - known_sites
    if unknown_sites:
        raise ValueError(f"Recorded sites {sorted(unknown_sites)} do not match substrate atom order: {smiles}")
    molecule = Chem.RemoveHs(molecule)
    rdDepictor.Compute2DCoords(molecule)
    return dict(substrate=smiles, molecule=molecule, sites=sites)


substrate_site_names = {}


def profile_site_label(profile):
    site = profile["substrate_site"]
    substrate = profile.get("substrate") or profile.get("reactant_row", {}).get("substrate")
    name = substrate_site_names.get((substrate, site))
    return name if name else str(site)


substrate_sites = defaultdict(set)
for row in jobs:
    if row.get("substrate") and row.get("substrate_site") is not None:
        substrate_sites[row["substrate"]].add(row["substrate_site"])
substrate_keys = [substrate_site_key(smiles, sites) for smiles, sites in sorted(substrate_sites.items())]
if substrate_keys:
    display(Markdown("## Substrate reaction sites"))
    palette = ((0.55, 0.75, 1.0), (1.0, 0.7, 0.35), (0.65, 0.85, 0.6), (0.85, 0.65, 0.9))
    drawings, legends, highlights, colors = [], [], [], []
    for key in substrate_keys:
        for index, site in enumerate(key["sites"]):
            substrate_site_names[(key["substrate"], site["substrate_site"])] = site["name"]
            drawings.append(Chem.Mol(key["molecule"]))
            legends.append(f"Site {site['name']}")
            highlights.append(site["atoms"])
            colors.append({atom: palette[index % len(palette)] for atom in site["atoms"]})
    options = Draw.MolDrawOptions()
    options.legendFontSize = 20
    display(Draw.MolsToGridImage(drawings, molsPerRow=2, subImgSize=(380, 300),
                                legends=legends, highlightAtomLists=highlights,
                                highlightAtomColors=colors, highlightBondLists=[[] for _ in drawings],
                                drawOptions=options, useSVG=True))


`substrate_site` is the one-based substrate atom index used by rxflow. Within each ligand / reaction-pair / seating / binding-permutation group, select the lowest Gibbs free energy among successful NumFreq results for each of reactant, TS and product. Endpoint selection includes all successful conformers, regardless of `retained`. A profile appears as soon as all three states have a successful result with a Gibbs energy; pending/running candidates do not hide it. Profiles with active jobs may change as more results finish. Failed rows are excluded even if they contain energies. All complete resonance/reaction-pair pathways are drawn separately, even when they share a site and binding permutation. For example, ligand 11 has O- and N-protonation pathways at each site under binding permutation 1. Legends identify the pair and resonance so both remain visible.

In [ ]:
# Select the lowest successful NumFreq Gibbs energy for each state in each group.
keys = ("ligand_id", "pair_index", "seating", "binding_permutation")
groups = defaultdict(list)
for row in jobs:
    key = tuple(row[column] for column in keys)
    groups[key].append(row)

# Encounter minima use the same active energy model and matching binding arrangement.
encounter_minima = {}
for row in encounter_jobs:
    if row["status"] != "succeeded" or row["gibbs_free_energy_eh"] is None:
        continue
    identity = (row["ligand_id"], row["binding_permutation"])
    previous = encounter_minima.get(identity)
    if previous is None or (row["gibbs_free_energy_eh"], row["id"]) < (previous["gibbs_free_energy_eh"], previous["id"]):
        encounter_minima[identity] = row

profiles = []
for key, all_rows in groups.items():
    rows = [row for row in all_rows if row["status"] == "succeeded"
            and row["gibbs_free_energy_eh"] is not None]
    reactants = [r for r in rows if r["endpoint"] == "reactant"]
    products = [r for r in rows if r["endpoint"] == "product"]
    transition_states = [r for r in rows if r["endpoint"] == "TS"]
    if not (reactants and products and transition_states):
        continue
    reactant = min(reactants, key=lambda row: row["gibbs_free_energy_eh"])
    product = min(products, key=lambda row: row["gibbs_free_energy_eh"])
    ts = min(transition_states, key=lambda row: row["gibbs_free_energy_eh"])
    encounter = encounter_minima.get((reactant["ligand_id"], reactant["binding_permutation"]))
    profiles.append(dict(zip(keys, key), encounter=encounter["gibbs_free_energy_eh"] if encounter else None,
                         encounter_job_id=encounter["id"] if encounter else None, reactant=reactant["gibbs_free_energy_eh"],
                         TS=ts["gibbs_free_energy_eh"], product=product["gibbs_free_energy_eh"],
                         reactant_row=reactant, product_row=product, ts_row=ts,
                         active_jobs=sum(row["status"] in ("pending", "running") for row in all_rows),
                         binding_mode=reactant["binding_mode"],
                         coordination_isomer=reactant["coordination_isomer"],
                         substrate_site=reactant["substrate_site"],
                         resonance=reactant["resonance"]))
print(f"Reaction profiles with energies in the active model: {len(profiles)}")
print(f"Profiles with candidates still pending/running: {sum(row['active_jobs'] > 0 for row in profiles)}")
print(f"Incomplete groups excluded: {len(groups) - len(profiles)}")

In [ ]:
# Count binding permutations, not TS candidates or repeated reaction pairs.
def identifier_list(values):
    return ", ".join("unknown" if value is None else str(value) for value in values) or "none"


def permutation_coverage(jobs, profiles):
    ligand_rows = defaultdict(list)
    reaction_groups = defaultdict(list)
    for row in jobs:
        ligand_rows[row["ligand_id"]].append(row)
        reaction_groups[tuple(row[column] for column in keys)].append(row)

    selected = defaultdict(list)
    for profile in profiles:
        selected[profile["ligand_id"]].append(profile)

    coverage = {}
    order = lambda value: (value is None, value if value is not None else 0)
    for ligand_id, rows in ligand_rows.items():
        sites = sorted({row["substrate_site"] for row in rows}, key=order)
        permutations = sorted({row["binding_permutation"] for row in rows}, key=order)
        site_coverage = []
        for site in sites:
            site_profiles = [profile for profile in selected[ligand_id]
                             if profile["substrate_site"] == site]
            shown = [p for p in permutations if any(profile["binding_permutation"] == p
                                                   for profile in site_profiles)]
            missing = [p for p in permutations if p not in shown]
            details = []
            for permutation in permutations:
                matching = [(key, group_rows) for key, group_rows in reaction_groups.items()
                            if key[0] == ligand_id and key[3] == permutation
                            and group_rows[0]["substrate_site"] == site]
                if not matching:
                    details.append((permutation, "No jobs recorded for this site/permutation."))
                for key, group_rows in matching:
                    unavailable = []
                    for role in ("reactant", "TS", "product"):
                        candidates = [row for row in group_rows if row["endpoint"] == role]
                        if any(row["status"] == "succeeded" and row["gibbs_free_energy_eh"] is not None
                               for row in candidates):
                            continue
                        selected_candidates = [row for row in candidates if row.get("singlepoint_selected")]
                        if selected_candidates:
                            counts = Counter("DFT " + row["singlepoint_status"] for row in selected_candidates)
                        else:
                            counts = Counter("sampling failed" if row["status"] == "failed"
                                             and row["stage"] == "sampling" else row["status"]
                                             for row in candidates)
                        status = ", ".join(f"{count} {name}" for name, count in sorted(counts.items()))
                        unavailable.append(f"{role}: {status or 'no jobs recorded'}")
                    if unavailable:
                        details.append((permutation, f"Pair {key[1]}, seating {key[2]} — missing plotted Gibbs energy: "
                                        + "; ".join(unavailable)))
            expected_paths = sum(key[0] == ligand_id and group_rows[0]["substrate_site"] == site
                                 for key, group_rows in reaction_groups.items())
            site_coverage.append(dict(site=site, expected=len(permutations), shown=shown,
                                      pathways_drawn=len(site_profiles), pathways_expected=expected_paths,
                                      missing=missing, details=details))
        coverage[ligand_id] = dict(sites=site_coverage, profiles=selected[ligand_id])
    return coverage


coverage_by_ligand = permutation_coverage(jobs, profiles)


def ligand_with_acceptor(ts_row):
    """Extract the ligand and preserve the actual TS acceptor through atom mapping."""
    if ts_row["acceptor"] is None:
        raise ValueError("TS row has no H-acceptor atom index")
    # Workflow XYZ indices are zero-based in ascending atom-map order.
    acceptor_map = ts_row["acceptor"] + 1
    parameters = Chem.SmilesParserParams()
    parameters.removeHs = False
    molecule = Chem.MolFromSmiles(ts_row["smiles"], parameters)
    if molecule is None:
        raise ValueError("Cannot parse the mapped reactant SMILES for this TS")
    coordinating_maps = {bond.GetBeginAtom().GetAtomMapNum() for bond in molecule.GetBonds()
                         if bond.GetBondType() == Chem.BondType.DATIVE}
    editable = Chem.RWMol(molecule)
    # Detach the metal coordination bonds, separating ligand from metal/substrate.
    for bond in molecule.GetBonds():
        if bond.GetBondType() == Chem.BondType.DATIVE:
            editable.RemoveBond(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
    fragments = Chem.GetMolFrags(editable.GetMol(), asMols=True, sanitizeFrags=True)
    ligand = next((fragment for fragment in fragments
                   if any(atom.GetAtomMapNum() == acceptor_map for atom in fragment.GetAtoms())), None)
    if ligand is None:
        raise ValueError(f"H-acceptor atom map {acceptor_map} is absent from the reactant graph")
    ligand = Chem.RemoveHs(ligand)
    highlight_atoms = [atom.GetIdx() for atom in ligand.GetAtoms()
                       if atom.GetAtomMapNum() == acceptor_map]
    if len(highlight_atoms) != 1:
        raise ValueError("Expected exactly one mapped ligand H-acceptor")
    symbol = ligand.GetAtomWithIdx(highlight_atoms[0]).GetSymbol()
    coordinating_atoms = [atom.GetIdx() for atom in ligand.GetAtoms()
                          if atom.GetAtomMapNum() in coordinating_maps]
    for atom in ligand.GetAtoms():
        atom.SetAtomMapNum(0)
        if atom.HasProp("atomNote"):
            atom.ClearProp("atomNote")
    rdDepictor.Compute2DCoords(ligand)
    return ligand, highlight_atoms, symbol, acceptor_map, coordinating_atoms


def successful_minimum(rows):
    valid = [row for row in rows if row["status"] == "succeeded"
             and row["gibbs_free_energy_eh"] is not None]
    return min(valid, key=lambda row: (row["gibbs_free_energy_eh"], row["id"])) if valid else None


def format_energy(value, absolute=False):
    return "missing" if value is None else (f"{value:.8f}" if absolute else f"{value:+.2f}")


def show_species_table(species_groups):
    table = ("<table><thead><tr><th>Species</th><th>Permutation</th><th>Native protonation site</th>"
             "<th>Charge</th><th>gxTB job</th><th>Conformer</th><th>DFT job</th><th>G plotted (Eh)</th><th>gxTB jobs</th></tr></thead><tbody>")
    for name, rows in species_groups:
        best = successful_minimum(rows)
        first = rows[0] if rows else {}
        counts = Counter(row["status"] for row in rows)
        status = ", ".join(f"{count} {state}" for state, count in sorted(counts.items())) or "no jobs recorded"
        values = (name, first.get("binding_permutation", "—"),
                  first.get("protonation_site", first.get("seating") if first.get("endpoint") == "protonated" else "—"),
                  first.get("charge", "—"), best["id"] if best else "missing",
                  best["conformer"] if best else "missing",
                  best.get("singlepoint_job_id", "—") if best else "missing",
                  format_energy(best["gibbs_free_energy_eh"] if best else None, absolute=True), status)
        table += "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in values) + "</tr>"
    display(HTML(table + "</tbody></table>"))


def proton_transfer_barriers(profile, limit=25.0):
    """Forward from the lower encounter/reactant energy; reverse from product."""
    encounter = profile.get("encounter")
    use_encounter = encounter is not None and encounter < profile["reactant"]
    reference = encounter if use_encounter else profile["reactant"]
    forward = (profile["TS"] - reference) * HARTREE_TO_KCAL
    bound_forward = (profile["TS"] - profile["reactant"]) * HARTREE_TO_KCAL
    reverse = (profile["TS"] - profile["product"]) * HARTREE_TO_KCAL
    # Allow only floating-point conversion noise at the inclusive threshold.
    qualifies = forward <= limit + 1e-9 and reverse <= limit + 1e-9
    return dict(forward_barrier_kcal_mol=forward, reverse_barrier_kcal_mol=reverse,
                bound_forward_barrier_kcal_mol=bound_forward,
                forward_reference_gibbs_eh=reference,
                forward_reference_state="encounter" if use_encounter else "reactant",
                both_barriers_within_limit=qualifies)


def pathway_grid_legend(profile, symbol, acceptor_map, relative_reactant_kcal_mol):
    """Show encounter-aware forward and product-referenced reverse barriers."""
    barriers = proton_transfer_barriers(profile, BARRIER_LIMIT_KCAL)
    return (
        f"Site {profile_site_label(profile)}, perm {profile['binding_permutation']}, "
        f"pair {profile['pair_index']}, res {profile['resonance']}\n"
        f"H-acceptor: {symbol} (atom map {acceptor_map})\n"
        f"Forward {barriers['forward_barrier_kcal_mol']:.2f}; "
        f"reverse {barriers['reverse_barrier_kcal_mol']:.2f} kcal/mol\n"
        f"Relative reactant G: {format_energy(relative_reactant_kcal_mol)} kcal/mol"
    )


def barrier_pathway_color(site_index, qualifies):
    """Use distinct high-contrast hues for each site and barrier class."""
    palettes = [("#0072B2", "#D55E00"), ("#009E73", "#AA4499"),
                ("#332288", "#CC6677"), ("#117733", "#994F00")]
    return palettes[site_index % len(palettes)][0 if qualifies else 1]


def energy_diagram(ligand_id, compared, reference_rows, acid_energy):
    """Use one absolute reference per ligand; keep unavailable levels and every native site."""
    extended = acid_energy is not None
    state_keys = ["protonated_start", "deprotonated_start", "encounter", "reactant", "TS", "product"] if extended else ["encounter", "reactant", "TS", "product"]
    labels = (["Protonated LM\n+ substrate + acetate", "Deprotonated LM\n+ substrate + AcOH",
               "Encounter complex\n+ AcOH", "Deprotonated bound complex\n+ AcOH", "TS\n+ AcOH", "Product\n+ AcOH"]
              if extended else ["Encounter complex", "Reactant", "TS", "Product"])
    ligand_refs = [row for row in reference_rows if row["ligand_id"] == ligand_id]
    include_bound = extended and any(row.get("protonated_bound_requested", False)
                                    or row.get("protonated_bound_with_acetate_gibbs_eh") is not None
                                    for row in ligand_refs)
    if include_bound:
        state_keys.insert(2, "protonated_bound")
        labels.insert(2, "Protonated bound complex\n+ acetate")
        # Association and ligand deprotonation can occur in either order.
        edges = [("protonated_start", "deprotonated_start"), ("deprotonated_start", "encounter"), ("encounter", "reactant"),
                 ("protonated_start", "protonated_bound"), ("protonated_bound", "reactant"),
                 ("reactant", "TS"), ("TS", "product")]
    else:
        edges = list(zip(state_keys, state_keys[1:]))
    linked = defaultdict(list)
    for row in ligand_refs:
        linked[tuple(row[key] for key in keys)].append(row)
    starts = [row["protonated_start_gibbs_eh"] for row in ligand_refs
              if row["protonated_start_gibbs_eh"] is not None] if extended else []
    if starts:
        reference = min(starts)
        reference_label = "lowest completed protonated LM + substrate + acetate"
    else:
        reference = min((row["reactant"] for row in compared), default=None)
        if extended and reference is not None:
            reference += acid_energy
        reference_label = ("lowest bound reactant + acetic acid (protonated-start reference unavailable)"
                           if extended else "lowest reactant (acetic-acid reference unavailable)")
    routes = []
    for profile in compared:
        matches = linked[tuple(profile[key] for key in keys)]
        for link in matches or [None]:
            energies = {state: profile[state] + (acid_energy if extended else 0)
                        for state in ("reactant", "TS", "product")}
            missing = []
            encounter_energy = link.get("encounter_gibbs_eh") if link else profile.get("encounter")
            energies["encounter"] = (encounter_energy + (acid_energy if extended else 0)
                                     if encounter_energy is not None else None)
            route_profile = dict(profile, encounter=encounter_energy)
            if energies["encounter"] is None:
                missing.append("relaxed encounter complex")
            if extended:
                energies.update(protonated_start=link["protonated_start_gibbs_eh"] if link else None,
                                deprotonated_start=link["deprotonated_start_gibbs_eh"] if link else None)
                if include_bound:
                    energies["protonated_bound"] = link.get("protonated_bound_with_acetate_gibbs_eh") if link else None
                    if energies["protonated_bound"] is None:
                        missing.append("protonated bound reactant + acetate")
                if link is None:
                    missing.append("No precursor link recorded for this pair/permutation/seating")
                else:
                    for field, name in (("protonated_complex_gibbs_eh", "protonated ligand–metal complex"),
                                        ("deprotonated_complex_gibbs_eh", "deprotonated ligand–metal complex"),
                                        ("substrate_gibbs_eh", "free fully protonated substrate"),
                                        ("acetate_gibbs_eh", "acetate")):
                        if link[field] is None:
                            missing.append(name)
            relative = {state: (energies[state] - reference) * HARTREE_TO_KCAL
                        if energies.get(state) is not None and reference is not None else None
                        for state in state_keys}
            routes.append(dict(profile=route_profile, link=link, energies=energies, relative=relative,
                               protonation_site=link["protonation_site"] if link else None,
                               missing=missing, **proton_transfer_barriers(route_profile, BARRIER_LIMIT_KCAL)))
    return dict(state_keys=state_keys, labels=labels, edges=edges, reference=reference,
                reference_label=reference_label, routes=routes, extended=extended)


## Which failed TS paths start from low-energy reactants?

Each pathway group (ligand / pair / seating / binding permutation) is represented by its **lowest successful reactant Gibbs-energy conformer**, including conformers outside the retained electronic-energy set. The energy zero is the lowest available reactant for the same ligand, substrate, charge and multiplicity **across all groups, including failed paths**. This is separate from the precursor-based zero on the reaction diagrams.

A group has TS coverage if **any candidate or NEB recovery succeeded**. A failed candidate does not count as a failed path when another succeeds. Pending/running recovery is shown as active. Missing DFT single points do not make a converged TS a failed search. This section assesses TS coverage; product, precursor and singlepoint readiness are separate.

Change **`TS_FAILURE_WINDOW_KCAL`** in the setup cell (default **3 kcal/mol**). The cumulative-window table also compares the exact minima and windows of 1, 3, 5 and 10 kcal/mol. Detailed failed paths are ordered by reactant ΔG; higher-energy failures are collapsed below. Every recorded pair/resonance and binding permutation is preserved. Some groups may share reactant minima, so these are pathway counts rather than independent conformer populations. Reactant energies indicate which failures may matter thermodynamically; they do not establish actual pathway visitation rates.


In [ ]:
import math


def reactant_ts_coverage(jobs, window_kcal=3.0):
    """Rank each group's reactant minimum and count TS outcomes after recovery."""
    if not math.isfinite(window_kcal) or window_kcal < 0:
        raise ValueError("The reactant energy window must be finite and nonnegative")
    grouped = defaultdict(list)
    for row in jobs:
        grouped[tuple(row[column] for column in keys)].append(row)

    report = []
    for key, rows in grouped.items():
        reactants = [row for row in rows if row["endpoint"] == "reactant"
                     and row["status"] == "succeeded"]
        available = [row for row in reactants if row.get("gibbs_free_energy_eh") is not None
                     and math.isfinite(row["gibbs_free_energy_eh"])]
        reactant = min(available, key=lambda row: (row["gibbs_free_energy_eh"], row["id"])) if available else None
        ts_rows = [row for row in rows if row["endpoint"] == "TS"]
        # A completed gxTB TS remains a success even when its DFT SP is missing.
        if any(row["status"] == "succeeded" for row in ts_rows):
            outcome = "complete"
        elif any(row["status"] == "running" for row in ts_rows):
            outcome = "running"
        elif any(row["status"] == "pending" for row in ts_rows):
            outcome = "pending"
        else:
            outcome = "failed" if ts_rows else "not_prepared"
        neb = [row for row in ts_rows if row.get("ts_strategy") == "neb"]
        terminal_rows = ([row for row in ts_rows if row["status"] == "succeeded"]
                         if outcome == "complete" else neb or ts_rows)
        terminal = max(terminal_rows, key=lambda row: (row.get("updated_at") or "", row["id"]), default=None)
        metadata = reactant or rows[0]
        if reactant is not None:
            energy_reason = ""
        elif any(row.get("gxtb_gibbs_free_energy_eh") is not None for row in reactants):
            energy_reason = "Selected reactant DFT single point unavailable"
        else:
            energy_reason = "No successful reactant Gibbs energy in the active model"
        report.append(dict(
            zip(keys, key), substrate_site=metadata.get("substrate_site"),
            resonance=metadata.get("resonance"), substrate=metadata.get("substrate"),
            outcome=outcome, reactant_row=reactant,
            reactant_gibbs_eh=reactant["gibbs_free_energy_eh"] if reactant else None,
            reactant_job_id=reactant["id"] if reactant else None,
            reactant_conformer=reactant.get("conformer") if reactant else None,
            reference_key=(key[0], metadata.get("substrate"), metadata.get("charge"),
                           metadata.get("multiplicity")),
            terminal_job_id=terminal["id"] if terminal else None,
            terminal_stage=terminal.get("stage") if terminal else None,
            terminal_reason=("Validated TS available" if outcome == "complete" else
                             (terminal.get("error") or terminal["status"]) if terminal else "No TS jobs recorded"),
            reactant_energy_reason=energy_reason))

    minima = {}
    for row in report:
        if row["reactant_gibbs_eh"] is not None:
            ref = row["reference_key"]
            minima[ref] = min(minima.get(ref, math.inf), row["reactant_gibbs_eh"])
    for row in report:
        energy = row["reactant_gibbs_eh"]
        row["delta_reactant_kcal_mol"] = ((energy - minima[row["reference_key"]]) * HARTREE_TO_KCAL
                                          if energy is not None else None)
        delta = row["delta_reactant_kcal_mol"]
        row["within_window"] = delta is not None and delta <= window_kcal + 1e-9
    report.sort(key=lambda row: (row["delta_reactant_kcal_mol"] is None,
                                 row["delta_reactant_kcal_mol"] or 0.,
                                 *(str(row[column]) for column in keys)))

    def counts(rows):
        states = Counter(row["outcome"] for row in rows)
        return dict(total=len(rows), complete=states["complete"], failed=states["failed"],
                    active=states["running"] + states["pending"], not_prepared=states["not_prepared"])

    ranked = [row for row in report if row["delta_reactant_kcal_mol"] is not None]
    windows = []
    for limit in [*sorted({0., 1., 3., 5., 10., float(window_kcal)}), None]:
        matching = [row for row in ranked if limit is None or row["delta_reactant_kcal_mol"] <= limit + 1e-9]
        windows.append(dict(limit_kcal_mol=limit, **counts(matching)))

    ligands = []
    for ligand in sorted({row["ligand_id"] for row in report}, key=str):
        rows = [row for row in report if row["ligand_id"] == ligand]
        low = [row for row in rows if row["within_window"]]
        lowest = [row for row in rows if row["delta_reactant_kcal_mol"] is not None
                  and row["delta_reactant_kcal_mol"] <= 1e-9]
        failed = [row["delta_reactant_kcal_mol"] for row in rows
                  if row["outcome"] == "failed" and row["delta_reactant_kcal_mol"] is not None]
        ligands.append(dict(ligand_id=ligand, ranked=sum(row["reactant_gibbs_eh"] is not None for row in rows),
                            unranked=sum(row["reactant_gibbs_eh"] is None for row in rows),
                            minimum_failed=sum(row["outcome"] == "failed" for row in lowest),
                            minimum_groups=len(lowest), lowest_failed_delta=min(failed, default=None),
                            **counts(low)))
    return dict(paths=report, windows=windows, ligands=ligands, totals=counts(report),
                unranked=[row for row in report if row["reactant_gibbs_eh"] is None],
                low_energy_failed=[row for row in report if row["within_window"] and row["outcome"] == "failed"])


def reactant_coverage_table(headers, rows):
    """Plain, escaped tables with no colored shading."""
    head = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
                   for row in rows)
    return f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"


def failed_reactant_table(rows):
    headers = ["Ligand", "Site", "Permutation", "Pair", "Seating", "Resonance", "Reactant ΔG (kcal/mol)",
               "Reactant job / conformer", "Terminal TS job / stage", "Final reason"]
    values = [(row["ligand_id"], row["substrate_site"], row["binding_permutation"], row["pair_index"],
               row["seating"], row["resonance"], format_energy(row["delta_reactant_kcal_mol"]),
               f"{row['reactant_job_id']} / {row['reactant_conformer']}",
               f"{row['terminal_job_id']} / {row['terminal_stage']}", row["terminal_reason"]) for row in rows]
    return reactant_coverage_table(headers, values)


In [ ]:
reactant_ts_analysis = reactant_ts_coverage(jobs, TS_FAILURE_WINDOW_KCAL)
reactant_ts_report = reactant_ts_analysis["paths"]  # One row per pathway group; available for export.
low_energy_failed_paths = reactant_ts_analysis["low_energy_failed"]
relative_reactant_by_group = {tuple(row[key] for key in keys): row["delta_reactant_kcal_mol"]
                              for row in reactant_ts_report}

display(Markdown("## TS coverage near the reactant minima"))
display(Markdown(f"**Energy model: {inputs['energy_model']}.** ΔG is measured from each ligand's lowest "
                 "available bound-reactant Gibbs energy, including groups with failed TS searches. "
                 "Comparisons keep substrate, charge and multiplicity the same. The common AcOH term cancels. "
                 "These are relative-energy priorities, not predicted populations or reaction rates."))
totals = reactant_ts_analysis["totals"]
display(Markdown(f"**{totals['failed']} terminal failed groups; {totals['active']} still active; "
                 f"{totals['not_prepared']} without TS jobs.** "
                 f"**{len(low_energy_failed_paths)} terminal failed groups lie within "
                 f"{TS_FAILURE_WINDOW_KCAL:g} kcal/mol of their reactant minimum.**"))

window_rows = []
for row in reactant_ts_analysis["windows"]:
    label = "All ranked groups" if row["limit_kcal_mol"] is None else f"≤ {row['limit_kcal_mol']:g}"
    fraction = f"{100 * row['failed'] / row['total']:.1f}%" if row["total"] else "—"
    window_rows.append((label, row["total"], row["complete"], row["failed"], row["active"], row["not_prepared"], fraction))
display(HTML(reactant_coverage_table(
    ["Reactant window (kcal/mol)", "Groups", "Validated TS", "Terminal failed", "Active", "No TS jobs", "Failed / groups"],
    window_rows)))

display(Markdown(f"### Per ligand: reactants within {TS_FAILURE_WINDOW_KCAL:g} kcal/mol"))
ligand_rows = [(row["ligand_id"], row["ranked"], f"{row['minimum_failed']} / {row['minimum_groups']}",
                row["total"], row["complete"], row["failed"], row["active"], row["not_prepared"],
                format_energy(row["lowest_failed_delta"]), row["unranked"])
               for row in reactant_ts_analysis["ligands"]]
display(HTML(reactant_coverage_table(
    ["Ligand", "Groups with reactant G", "Failed / groups at minimum", "Groups in window", "Validated TS", "Terminal failed",
     "Active", "No TS jobs", "Lowest failed ΔG (kcal/mol)", "Unranked groups"], ligand_rows)))

display(Markdown(f"### Failed paths within {TS_FAILURE_WINDOW_KCAL:g} kcal/mol — lowest reactant energy first"))
if low_energy_failed_paths:
    display(HTML(failed_reactant_table(low_energy_failed_paths)))
else:
    display(Markdown("No terminal failed TS paths in this energy window among groups with available reactant energies."))

other_failed = [row for row in reactant_ts_report if row["outcome"] == "failed"
                and row["delta_reactant_kcal_mol"] is not None and not row["within_window"]]
if other_failed:
    display(HTML(f"<details><summary>{len(other_failed)} failed paths above the selected energy window</summary>"
                 + failed_reactant_table(other_failed) + "</details>"))

unranked = reactant_ts_analysis["unranked"]
if unranked:
    display(Markdown(f"**{len(unranked)} groups have no reactant energy in the active model and cannot be ranked.** "
                     "They are excluded from energy-window counts. In a partial DFT view, the minimum is only "
                     "the lowest available corrected energy; missing single points may change this ranking."))
    unranked_rows = [(row["ligand_id"], row["pair_index"], row["seating"], row["binding_permutation"],
                      row["outcome"], row["reactant_energy_reason"], row["terminal_reason"])
                     for row in unranked]
    display(HTML(reactant_coverage_table(
        ["Ligand", "Pair", "Seating", "Permutation", "TS outcome", "Missing reactant energy", "TS status / reason"],
        unranked_rows)))


In [ ]:
display(Markdown("## Shared acid/base reference calculations"))
show_species_table([(name, [row for row in reference_jobs if row["ligand_id"] == name])
                    for name in ("acetic_acid", "acetate")])
if any(row["status"] in ("pending", "running") for row in precursor_jobs + reference_jobs + bound_reactant_jobs + encounter_jobs):
    display(Markdown("**Calculations are still active.** The notebook uses the lowest successful NumFreq result available now; rerun all cells to refresh the minima and references."))


# One report row per reaction pair/seating/permutation, independent of shared precursor entries.
barrier_report = sorted([dict(**{key: profile[key] for key in keys}, substrate_site=profile["substrate_site"],
                      resonance=profile["resonance"], **proton_transfer_barriers(profile, BARRIER_LIMIT_KCAL))
                  for profile in profiles], key=lambda row: (
                      row["forward_barrier_kcal_mol"], row["reverse_barrier_kcal_mol"],
                      row["ligand_id"], row["binding_permutation"], row["pair_index"], row["seating"]))
qualifying_pathways = [row for row in barrier_report if row["both_barriers_within_limit"]]
display(Markdown(f"## Proton-transfer pathways with both barriers ≤ {BARRIER_LIMIT_KCAL:g} kcal/mol"))
display(Markdown(f"**{len(qualifying_pathways)} / {len(barrier_report)} complete pathways qualify.** "
                 "Forward = TS − the lower of encounter and bound reactant; reverse = TS − bound product. Ranked by forward barrier. "
                 "Contrasting plot colors distinguish each site and barrier class. The tables and ligand-grid captions use plain text."))
if qualifying_pathways:
    headers = ["Ligand", "Site", "Permutation", "Pair", "Seating", "Resonance", "Forward zero", "Forward (kcal/mol)", "Reverse (kcal/mol)"]
    table = "<table><thead><tr>" + "".join(f"<th>{header}</th>" for header in headers) + "</tr></thead><tbody>"
    for row in qualifying_pathways:
        values = [row["ligand_id"], row["substrate_site"], row["binding_permutation"], row["pair_index"],
                  row["seating"], row["resonance"], row["forward_reference_state"], f"{row['forward_barrier_kcal_mol']:.2f}",
                  f"{row['reverse_barrier_kcal_mol']:.2f}"]
        table += "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in values) + "</tr>"
    display(HTML(table + "</tbody></table>"))
else:
    display(Markdown("No complete pathways satisfy both barrier limits yet."))


In [ ]:
def plot_energy_diagram(ligand_id, diagram, energy_progress, site_order):
    """Draw the selected energy model with compact pathway legends."""
    from matplotlib.lines import Line2D
    from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea
    from matplotlib.ticker import AutoMinorLocator, MaxNLocator

    state_keys, labels = diagram["state_keys"], diagram["labels"]
    x = list(range(len(state_keys)))
    half_width = 0.23
    site_indices = {site: index for index, site in enumerate(site_order)}
    # Different native precursor protonations share a reaction-path style.
    # Their separate levels and routes remain in the diagram and numerical table.
    path_keys = list(dict.fromkeys((route["profile"]["binding_permutation"],
                                   route["profile"]["pair_index"], route["profile"]["seating"])
                                  for route in diagram["routes"]))
    line_styles = {key: ("-", "--", ":", "-.")[index % 4] for index, key in enumerate(path_keys)}
    height = max(7.2, 2.8 + 0.34 * len(path_keys))
    fig, ax = plt.subplots(figsize=(15 if diagram["extended"] else 12, height), layout="constrained")
    fig.set_facecolor("white")
    ax.set_facecolor("white")
    handles, seen = [], set()
    for route in diagram["routes"]:
        row = route["profile"]
        relative = [route["relative"][state] for state in state_keys]
        site, permutation = row["substrate_site"], row["binding_permutation"]
        color = barrier_pathway_color(site_indices[site], route["both_barriers_within_limit"])
        path_key = (permutation, row["pair_index"], row["seating"])
        style = line_styles[path_key]
        label = (f"Site {profile_site_label(row)} · perm {permutation} · pair {row['pair_index']}\n"
                 f"{route['forward_barrier_kcal_mol']:.1f} / {route['reverse_barrier_kcal_mol']:.1f}")
        if path_key not in seen:
            handles.append(Line2D([0], [0], color=color, linestyle=style, linewidth=1.8,
                                  marker="o", markersize=4, label=label))
            seen.add(path_key)
        for state, position, energy in zip(state_keys, x, relative):
            if energy is None:
                continue
            ax.hlines(energy, position - half_width, position + half_width,
                      color=color, linewidth=3.0, zorder=3)
            ax.scatter(position, energy, color=color, edgecolor="white", linewidth=0.7,
                       s=42 if state == "TS" else 30, zorder=4,
                       marker="D" if state == "TS" else ("^" if state == "protonated_bound" else "o"))
        for left, right in diagram["edges"]:
            before, after = state_keys.index(left), state_keys.index(right)
            if relative[before] is not None and relative[after] is not None:
                ax.plot([x[before] + half_width, x[after] - half_width],
                        [relative[before], relative[after]], color=color,
                        linestyle=style, linewidth=1.5, alpha=0.85, zorder=2)

    if energy_progress["active_mode"] == "dft":
        model = "PBE0/def2-TZVP-D4 · SMD(water) · gxTB thermal correction"
    else:
        model = "gxTB Gibbs free energies"
        if energy_progress.get("requested_mode") == "auto":
            model += (f" · DFT corrections ready: {energy_progress['ready']} / "
                      f"{energy_progress['selected_states']}")
    heading = HPacker(children=[
        TextArea(f"Ligand {ligand_id} · Substrate reaction profiles",
                 textprops=dict(fontsize=17, fontweight="semibold", color="#1D2939")),
        TextArea(model, textprops=dict(fontsize=11, color="#475467")),
    ], align="baseline", pad=0, sep=14)
    ax.add_artist(AnchoredOffsetbox(loc="lower left", child=heading, frameon=False,
                                   bbox_to_anchor=(0, 1.035), bbox_transform=ax.transAxes,
                                   borderpad=0, pad=0))
    ax.set_xticks(x, labels, rotation=30, ha="right", rotation_mode="anchor", fontsize=10)
    ax.tick_params(axis="x", length=0, pad=10, colors="#344054")
    ax.tick_params(axis="y", labelsize=10, colors="#344054")
    ax.set_xlim(-0.55, x[-1] + 0.55)
    ax.margins(y=0.12)
    ax.set_ylabel(r"$\Delta G$ (kcal mol$^{-1}$)", fontsize=12, labelpad=10)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=7))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.set_axisbelow(True)
    ax.grid(axis="y", which="major", color="#D0D5DD", linewidth=0.7, alpha=0.7)
    ax.grid(axis="y", which="minor", color="#EAECF0", linewidth=0.4, alpha=0.5)
    ax.axhline(0, color="#667085", linestyle=(0, (3, 3)), linewidth=0.9, zorder=1)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("bottom", "left"):
        ax.spines[spine].set_color("#D0D5DD")
    legend = ax.legend(handles=handles, title="Pathway\nForward / reverse (kcal/mol)",
                       fontsize=9, title_fontsize=10, frameon=False,
                       labelspacing=0.8, handlelength=2.4, borderaxespad=0,
                       bbox_to_anchor=(1.025, 1), loc="upper left")
    legend.get_title().set_color("#344054")
    return fig, ax


In [ ]:
ligand_ids = coverage_by_ligand
diagram_rows = []  # Numerical balanced-state results, also available for inspection/export.

for ligand_id in ligand_ids:
    group = [row for row in profiles if row["ligand_id"] == ligand_id]
    coverage = coverage_by_ligand[ligand_id]
    compared = coverage["profiles"]
    display(Markdown(f"### Ligand {ligand_id}: permutation coverage by site"))
    table = ("<table><thead><tr><th>Site</th><th>Expected permutations</th><th>Drawn</th>"
             "<th>Missing</th><th>Drawn IDs</th><th>Missing IDs</th><th>Pathways drawn / recorded</th></tr></thead><tbody>")
    for item in coverage["sites"]:
        site = identifier_list([item["site"]])
        table += (f"<tr><td>{escape(site)}</td><td>{item['expected']}</td>"
                  f"<td>{len(item['shown'])}</td><td>{len(item['missing'])}</td>"
                  f"<td>{escape(identifier_list(item['shown']))}</td>"
                  f"<td>{escape(identifier_list(item['missing']))}</td>"
                  f"<td>{item['pathways_drawn']} / {item['pathways_expected']}</td></tr>")
    display(HTML(table + "</tbody></table>"))
    details = [(item["site"], permutation, reason) for item in coverage["sites"]
               for permutation, reason in item["details"]]
    if details:
        table = ("<table><thead><tr><th>Site</th><th>Permutation</th>"
                 "<th>Why no complete profile?</th></tr></thead><tbody>")
        for site, permutation, reason in details:
            table += (f"<tr><td>{escape(identifier_list([site]))}</td>"
                      f"<td>{escape(identifier_list([permutation]))}</td>"
                      f"<td>{escape(reason)}</td></tr>")
        display(HTML(table + "</tbody></table>"))
    display(Markdown("**Precursor minima and calculation coverage**"))
    species_groups = defaultdict(list)
    substrate_ids = {row["substrate_id"] for row in precursor_references if row["ligand_id"] == ligand_id}
    for row in precursor_jobs + bound_reactant_jobs + encounter_jobs:
        if row["ligand_id"] == ligand_id or row["ligand_id"] in substrate_ids:
            identity = (row["ligand_id"], row["binding_permutation"], row["seating"], row["endpoint"],
                        row["pair_index"] if row["endpoint"] == "protonated_bound" else 0,
                        row.get("protonation_site", 0))
            species_groups[identity].append(row)
    if species_groups:
        show_species_table([(f"{identity[0]}: {identity[3]}" +
                             (f", pair {identity[4]}, seating {identity[2]}" if identity[3] == "protonated_bound" else ""), rows)
                            for identity, rows in sorted(species_groups.items())])
    else:
        display(Markdown("No precursor calculations are recorded for this ligand yet."))
    if not compared:
        display(Markdown("No complete reactant–TS–product profiles yet."))
        continue
    diagram = energy_diagram(ligand_id, compared, precursor_references, acid_gibbs_eh)
    reference = diagram["reference"]
    state_keys, labels = diagram["state_keys"], diagram["labels"]
    x = list(range(len(state_keys)))
    display(Markdown(f"**Common reference:** {diagram['reference_label']}; G = {reference:.8f} Eh. "
                     "All plotted sites and permutations for this ligand share this zero."))
    unavailable = [route for route in diagram["routes"] if route["missing"]]
    if unavailable:
        table = ("<table><thead><tr><th>Site</th><th>Permutation</th><th>Pair</th>"
                 "<th>Unavailable precursor inputs (levels left blank)</th></tr></thead><tbody>")
        for route in unavailable:
            row = route["profile"]
            table += (f"<tr><td>{row['substrate_site']}</td><td>{row['binding_permutation']}</td>"
                      f"<td>{row['pair_index']}</td><td>{escape('; '.join(route['missing']))}</td></tr>")
        display(HTML(table + "</tbody></table>"))

    display(Markdown(f"### Ligand {ligand_id}: balanced pathway comparison (kcal/mol)"))
    headers = ["Site", "Permutation", "Pair", "Resonance", "Native protonation site"]
    headers += [label.replace("\n", " ") for label in labels]
    headers += ["Forward zero", "Forward barrier", "Reverse barrier", f"Both ≤ {BARRIER_LIMIT_KCAL:g}?", "Reaction energy"]
    table = "<table><thead><tr>" + "".join(f"<th>{escape(header)}</th>" for header in headers) + "</tr></thead><tbody>"
    for route in sorted(diagram["routes"], key=lambda route: route["forward_barrier_kcal_mol"]):
        row, relative = route["profile"], route["relative"]
        barrier = route["forward_barrier_kcal_mol"]
        reverse_barrier = route["reverse_barrier_kcal_mol"]
        reaction = (row["product"] - row["reactant"]) * HARTREE_TO_KCAL
        values = [row["substrate_site"], row["binding_permutation"], row["pair_index"], row["resonance"],
                  route["protonation_site"] if route["protonation_site"] is not None else "—"]
        values += [format_energy(relative[state]) for state in state_keys]
        values += [route["forward_reference_state"], format_energy(barrier), format_energy(reverse_barrier),
                   "yes" if route["both_barriers_within_limit"] else "no", format_energy(reaction)]
        table += "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in values) + "</tr>"
        diagram_rows.append(dict(ligand_id=ligand_id, substrate_site=row["substrate_site"],
            binding_permutation=row["binding_permutation"], pair_index=row["pair_index"],
            seating=row["seating"], resonance=row["resonance"], protonation_site=route["protonation_site"],
            reference_eh=reference, reference_label=diagram["reference_label"], energy_model=inputs["energy_model"],
            absolute_gibbs_eh=route["energies"], relative_kcal_mol=relative,
            forward_barrier_kcal_mol=barrier, reverse_barrier_kcal_mol=reverse_barrier,
            forward_reference_state=route["forward_reference_state"],
            forward_reference_gibbs_eh=route["forward_reference_gibbs_eh"],
            barrier_from_reactant_kcal_mol=route["bound_forward_barrier_kcal_mol"],
            relative_reactant_kcal_mol=relative_reactant_by_group[tuple(row[key] for key in keys)],
            both_barriers_within_limit=route["both_barriers_within_limit"], reaction_energy_kcal_mol=reaction,
            singlepoint_job_ids={name: row[row_key].get("singlepoint_job_id")
                                 for name, row_key in (("reactant", "reactant_row"), ("TS", "ts_row"), ("product", "product_row"))},
            precursor_job_ids={name: route["link"].get(name) if route["link"] else None
                               for name in ("protonated_job_id", "deprotonated_job_id", "substrate_job_id", "protonated_bound_job_id", "encounter_job_id")}))
    display(HTML(table + "</tbody></table>"))

    fig, ax = plot_energy_diagram(
        ligand_id, diagram, singlepoint_progress,
        [item["site"] for item in coverage["sites"]],
    )
    plt.show()

    # Keep grid captions black and report the threshold in the section heading.
    display(Markdown("**Ligand atoms for each plotted pathway:** blue = metal-binding; "
                     "red = H-acceptor; purple = both. Grid legends give forward and reverse "
                     "barriers and relative reactant G in kcal/mol. Forward barriers use the lower of the encounter and bound reactant. Relative reactant G "
                     "is measured from the lowest deprotonated bound reactant for this ligand across "
                     "comparable groups, including groups without a TS, in the active energy model."))
    for qualifying in (True, False):
        grid_profiles = [profile for profile in compared
                         if proton_transfer_barriers(profile, BARRIER_LIMIT_KCAL)["both_barriers_within_limit"] == qualifying]
        if not grid_profiles:
            continue
        heading = (f"**Both barriers ≤ {BARRIER_LIMIT_KCAL:g} kcal/mol**"
                   if qualifying else f"**Other pathways — at least one barrier > {BARRIER_LIMIT_KCAL:g} kcal/mol**")
        display(Markdown(heading))
        ligand_mols, highlight_atom_lists, highlight_atom_colors, ligand_legends = [], [], [], []
        for profile in sorted(grid_profiles, key=lambda row: (proton_transfer_barriers(row)["forward_barrier_kcal_mol"], row["substrate_site"] or 0,
                                                              row["binding_permutation"] or 0,
                                                              row["pair_index"], row["seating"])):
            ligand, acceptor_atoms, symbol, acceptor_map, coordinating_atoms = ligand_with_acceptor(profile["ts_row"])
            atom_colors = {atom: (0.25, 0.55, 1.0) for atom in coordinating_atoms}
            for atom in acceptor_atoms:
                atom_colors[atom] = (0.65, 0.35, 0.85) if atom in coordinating_atoms else (1.0, 0.4, 0.4)
            ligand_mols.append(ligand)
            highlight_atom_lists.append(sorted(atom_colors))
            highlight_atom_colors.append(atom_colors)
            legend = pathway_grid_legend(
                profile, symbol, acceptor_map,
                relative_reactant_by_group[tuple(profile[key] for key in keys)],
            )
            ligand_legends.append(legend)
        options = Draw.MolDrawOptions()
        options.legendFontSize = 14
        options.legendFraction = 0.30
        options.setLegendColour((0.0, 0.0, 0.0))
        display(Draw.MolsToGridImage(
            ligand_mols, molsPerRow=min(3, len(ligand_mols)), subImgSize=(380, 360),
            legends=ligand_legends, highlightAtomLists=highlight_atom_lists,
            highlightAtomColors=highlight_atom_colors, drawOptions=options,
            highlightBondLists=[[] for _ in ligand_mols], useSVG=True,
        ))

    # Render the structures for the completed pathway with the lowest reactant Gibbs energy.
    lowest = min(group, key=lambda row: row["reactant"])
    display(Markdown(
        f"### Lowest-reactant pathway: site {profile_site_label(lowest)}, "
        f"permutation {lowest['binding_permutation']}, pair {lowest['pair_index']}"
    ))
    for title, row_key in (("Reactant", "reactant_row"), ("Transition state", "ts_row"),
                           ("Product", "product_row")):
        structure = lowest[row_key]
        xyz_path = DB_PATH.resolve().parent / structure["xyz_file"]
        display(Markdown(f"**{title}** — $G = {structure['gibbs_free_energy_eh']:.8f}$ Eh"))
        if not xyz_path.is_file():
            print(f"XYZ not found: {xyz_path}")
            continue
        render_options = {"no_hy": True, "bo": False}
        if row_key == "ts_row":
            reacting_atoms = (structure["donor"], structure["hydrogen"], structure["acceptor"])
            if all(atom is not None for atom in reacting_atoms):
                donor, hydrogen, acceptor = (atom + 1 for atom in reacting_atoms)
                render_options["ts_bonds"] = [(donor, hydrogen), (hydrogen, acceptor)]
        display(xyzrender.render(xyz_path, **render_options))